# Jena Climate Forecasting

Predict the next **7 observations** of temperature from 144 past multivariate observations, using Conv1D and LSTM layers.

Original saved results: model MAE **0.452°C**, persistence baseline MAE **0.506°C**. These are historical results, not a fresh training run.


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from pathlib import Path


## 1. Load the dataset


In [ ]:
uri = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
extracted_dir = Path(keras.utils.get_file(
    origin=uri,
    fname="jena_climate_2009_2016.csv.zip",
    extract=True,
))
csv_files = sorted(extracted_dir.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError("No climate CSV found in the extracted dataset")
df = pd.read_csv(csv_files[0])
print("Dataset shape:", df.shape)
df.head()



## 2. Prepare features and chronological splits


In [ ]:
feature_df = df.drop(columns=["Date Time"])
feature_names = feature_df.columns.tolist()

temperature = df["T (degC)"].astype("float32").to_numpy()
features = feature_df.astype("float32").to_numpy()

temp_col = feature_names.index("T (degC)")

n = len(features)
num_train = int(0.5 * n)
num_val = int(0.25 * n)
num_test = n - num_train - num_val

print(num_train, num_val, num_test)

mean = features[:num_train].mean(axis=0)
std = features[:num_train].std(axis=0)

if np.any(std == 0):
    raise ValueError("A training feature has zero variance")
features = (features - mean) / std




## 3. Build supervised windows


In [ ]:
lookback = 144
horizon = 7
batch_size = 256

features_tf = tf.convert_to_tensor(features, dtype=tf.float32)
temperature_tf = tf.convert_to_tensor(temperature, dtype=tf.float32)

def make_dataset(start, end, shuffle=False):
    max_start = end - lookback - horizon + 1
    ds = tf.data.Dataset.range(start, max_start)

    if shuffle:
        ds = ds.shuffle(10000, seed=42)

    def get_window(i):
        x = features_tf[i : i + lookback]
        y = temperature_tf[i + lookback : i + lookback + horizon]
        return x, y
    ds = ds.map(get_window, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(0, num_train, shuffle=True)
val_ds = make_dataset(num_train, num_train + num_val)
test_ds = make_dataset(num_train + num_val, n)




## 4. Evaluate the persistence baseline


In [ ]:
def evaluate_naive(ds):
    total_abs_error = 0.0
    total_values = 0

    for x, y in ds:
        last_temp = x[:, -1, temp_col] * std[temp_col] + mean[temp_col]
        pred = tf.repeat(last_temp[:, None], repeats=horizon, axis=1)

        total_abs_error += tf.reduce_sum(tf.abs(y - pred)).numpy()
        total_values += np.prod(y.shape)
    return total_abs_error / total_values

naive_mae = evaluate_naive(test_ds)
print(f"Naive baseline MAE: {naive_mae:.3f} °C")




## 5. Build the Conv1D + LSTM model


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(lookback, features.shape[-1])),
    layers.Conv1D(32, 5, activation="relu"),
    layers.MaxPooling1D(2),
    layers.LSTM(32),
    layers.Dense(32, activation="relu"),
    layers.Dense(horizon),
])

model.compile(
    optimizer="rmsprop",
    loss="mae",
    metrics=["mae"],
)


model.summary()



## 6. Train with early stopping


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_mae",
        patience=3,
        restore_best_weights=True,
    )
]

history = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=callbacks,
)




## 7. Evaluate on the test split


In [ ]:
results = model.evaluate(test_ds, return_dict=True)

metric = float(results["mae"])
print(f"Test MAE: {metric:.3f} °C")


## 8. Learning curves


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["mae"], label="Train")
plt.plot(history.history["val_mae"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("MAE (°C)")
plt.title("Temperature forecasting learning curves")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 9. Example forecast

One example illustrates the seven forecast steps; it is not an aggregate measure of performance.


In [ ]:
x_batch, y_batch = next(iter(test_ds))
predicted = model(x_batch[:1], training=False).numpy()[0]
last_temp = float(x_batch[0, -1, temp_col] * std[temp_col] + mean[temp_col])
steps = np.arange(1, horizon + 1)
plt.plot(steps, y_batch[0].numpy(), "o-", label="Observed")
plt.plot(steps, predicted, "o-", label="Conv1D + LSTM")
plt.axhline(last_temp, linestyle="--", label="Persistence baseline")
plt.xlabel("Future observation step")
plt.ylabel("Temperature (°C)")
plt.xticks(steps)
plt.legend()
plt.show()


## Provenance and limitations

Developed for DATAX504 Assignment 5, time-series track. The original notebook disclosed Codex GPT-5.5 assistance with suggestions and explanations, followed by author review and execution. Codex assisted in extracting this route, correcting the download path for the inspected Keras version, documenting it and adding visualisations. Full training was not repeated during packaging.

Inputs are normalised using training statistics; targets stay in °C. Windows stay within their respective 50%/25%/25% chronological splits. Only training windows are shuffled. No global random seed was set in the original experiment. This is a seven-observation forecast, not a seven-day forecast.
